# OSHA Llama 3.1 Fine-Tuning (QLoRA + Learning Curve Experiment)

This notebook uses **Unsloth** to efficiently fine-tune **Llama 3.1 8B** on a free Colab T4 GPU.

**Pipeline:** Install → Load Model (4-bit) → Load Data → Train (QLoRA) → Evaluate → Export GGUF

In [1]:
%%capture
# 1. Install Unsloth and Dependencies
!pip install unsloth
!pip install --upgrade torch torchvision
!pip install xformers trl peft accelerate bitsandbytes

In [2]:
# 2. Load the Base Model (Llama 3.1 8B) in 4-bit (QLoRA)
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True  # This is the "Q" in QLoRA — reduces VRAM by 75%

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA adapters (tiny trainable weights on top of the frozen base)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

/usr/local/lib/python3.13/dist-packages/unsloth/_gpu_init.py:98: UserWarning: Unsloth: torchaudio cannot initialise against this torch and has been disabled for this process, so anything that needs it will report it as missing rather than crash at import. Install the matching wheel to restore it. Original error: Detected that PyTorch and TorchAudio were compiled with different CUDA versions. PyTorch has CUDA version 13.0 whereas TorchAudio has CUDA version 12.8. Please install the TorchAudio version that matches your PyTorch version.
  disable_torchaudio_if_cuda_mismatched()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0824 13:23:27.795000 3540 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0824 13:23:27.950000 3540 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:1989: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)


==((====))==  Unsloth 2026.8.19: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.13.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.8.19 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


### UPLOAD YOUR DATA NOW
1. Look at the left sidebar in Colab.
2. Click the Folder icon.
3. Click the Upload icon and upload your training file (e.g. `train_1k.jsonl`) AND `gold_eval_set.csv`.

In [3]:
# 3. Load Your Training Data
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

# === CHANGE THIS FILENAME for each run ===
TRAIN_FILE = "train_1k.jsonl"  # Options: train_1k.jsonl, train_3k.jsonl, train_5k.jsonl
# =========================================

dataset = load_dataset("json", data_files=TRAIN_FILE, split="train")
dataset = dataset.map(formatting_prompts_func, batched = True,)
print(f"Loaded {len(dataset)} training examples from {TRAIN_FILE}")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded 1000 training examples from train_1k.jsonl


In [4]:
# 4. Train the Model (SFT with QLoRA)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.796132
20,0.343413
30,0.218745
40,0.211949
50,0.210243
60,0.226826
70,0.220051
80,0.207547
90,0.218380
100,0.205554


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-125/tokenizer_config.json.


In [5]:
# 5. Evaluate on Gold Standard (200 rows)
import pandas as pd
import json

FastLanguageModel.for_inference(model)

system_prompt = """You are a construction safety analyst. Given an OSHA incident narrative, extract the following structured fields.
Respond with ONLY a valid JSON object. Do not include markdown formatting or explanation.

Fields:
- event_type: one of [fall_to_lower_level, struck_by_object, caught_in_between, struck_by_vehicle, electrocution, heat_exposure, fall_same_level, struck_against, fall_through_surface, contact_hot_substance, collapse_engulfment, trench_cave_in, explosion_ignition, chemical_exposure, equipment_overturned, other]
- injury_nature: one of [fracture, amputation, laceration, contusion_pain, burn, internal_injury, intracranial, crushing, heat_illness, dislocation, poisoning_toxic, sprain_strain_tear, heart_attack, other]
- body_part: one of [finger_hand, leg, head_brain, chest_trunk, multiple, foot_ankle, arm, back_spine, wrist, hip_pelvis, shoulder, body_systems, eye, neck, other]
- source_equipment: one of [ladder, scaffold, roof, power_tool, vehicle_heavy_equip, aerial_lift, electrical_source, structural, floor_ground_stair, pipe_duct, environmental, trench_excavation, nail_fastener, metal_material, rebar, debris, pole, machinery_equipment, building_materials_parts, bodily_motion, boxes_containers, tanks_vats, covers_lids, chemicals_fumes, window_opening, hose_cable, other]
- hospitalized: true or false
- amputation: true or false"""

df = pd.read_csv("gold_eval_set.csv")
results = []
correct = 0
total = 0

# === CHANGE THIS for each run ===
RUN_NAME = "1k"  # Options: 1k, 3k, 5k
# ================================

print(f"Evaluating {RUN_NAME} model on {len(df)} gold rows...\n")

for i, row in df.iterrows():
    convo = [{"role": "system", "content": system_prompt}, {"role": "user", "content": f"Narrative: {row['Final Narrative']}"}]
    inputs = tokenizer.apply_chat_template(convo, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=256, use_cache=True, do_sample=False)
    prompt_length = inputs.shape[1]
    raw_output = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True).strip()
    result_row = {"ID": row.get("ID", i), "Narrative": row["Final Narrative"], "Raw_LLM_Output": raw_output}
    try:
        prediction = json.loads(raw_output)
        fields = [("event_type", "TRUE_event"), ("injury_nature", "TRUE_nature"), ("body_part", "TRUE_body"), ("source_equipment", "TRUE_source")]
        for pred_key, true_col in fields:
            total += 1
            true_val = str(row[true_col]).strip().lower()
            # Fallback to automated mappings if the human TRUE_ column is empty
            if true_val == "" or true_val == "nan":
                fallback = true_col.replace("TRUE_", "mapped_")
                if fallback in row: true_val = str(row[fallback]).strip().lower()
            pred_val = str(prediction.get(pred_key, "")).strip().lower()
            result_row[f"pred_{pred_key}"] = prediction.get(pred_key, "")
            if pred_val == true_val:
                correct += 1
    except Exception:
        total += 4
        result_row["pred_event"] = "JSON_ERROR"
    results.append(result_row)
    if (i+1) % 20 == 0:
        print(f"Processed {i+1}/{len(df)}...")

output_csv = f"predictions_{RUN_NAME}.csv"
results_df = pd.DataFrame(results)
results_df.to_csv(output_csv, index=False)
print(f"\n========================================")
print(f"Fine-Tuned Accuracy: {(correct/total)*100:.1f}%")
print(f"Saved to {output_csv}")
print(f"========================================")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Evaluating 1k model on 200 gold rows...



Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Processed 20/200...


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

...

Processed 160/200...


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Processed 180/200...


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Processed 200/200...

Fine-Tuned Accuracy: 71.8%
Saved to predictions_1k.csv


In [ ]:
# 6. Export to GGUF for Local Deployment (Takes ~15-20 minutes)
# This fuses LoRA weights into the base model and quantizes to 4-bit GGUF
model.save_pretrained_gguf(
    "model",
    tokenizer,
    quantization_method = "q4_k_m",
)
print("GGUF export complete! Download the .gguf file from the Files sidebar.")